# TMJDLC — Javtiful Telegram Bot (Google Colab)

**Runtime:** Three Pyrogram MTProto sessions are used: the main bot handles Telegram updates and commands, its dedicated upload-only session handles videos, and a separate screenshot bot handles screenshot delivery. Neon PostgreSQL stores the durable queue, job history, worker leases, and bot settings. The bot is Javtiful-only and selects **720p automatically**.

Downloads use upload-queue backpressure: one video may wait for upload, then new downloads pause until the backlog clears. Run the cells from top to bottom. The source-sync cell resets the workspace to the latest `main` branch, including the dedicated video upload client and separate screenshot bot client. Install dependencies once per fresh runtime; after a runtime restart, sync and launch again.


## 1. Configure Colab Secrets

Add these six secrets in Colab's Secrets panel and enable notebook access:

- `TG_BOT_TOKEN` — main bot token from @BotFather
- `TG_SCREENSHOT_BOT_TOKEN` — token of a second bot created with @BotFather, used only for screenshot uploads
- `TG_API_ID` — Telegram API ID from my.telegram.org
- `TG_API_HASH` — Telegram API hash from my.telegram.org
- `TG_ADMIN_IDS` — numeric Telegram user ID(s); separate multiple IDs with commas
- `DATABASE_URL` — Neon PostgreSQL connection string

Do not paste credentials into notebook cells. Configure video and screenshot destination channels later from Telegram using `/setup` or the Settings panel. The main bot must be an administrator in the video destination. The screenshot bot must separately be an administrator in the screenshot destination channel. Configure a different token for each bot.

For a private video channel, run `/setupload` without an ID and forward a recent channel post to the main bot. For a private screenshot channel, run `/setimages` without an ID and forward a recent channel post to the main bot; the screenshot bot itself must already be a member/administrator of that channel. For public channels, use the channel's `@username`.

If a previously saved video destination becomes unresolvable at job time, TMJDLC sends the video to the requesting user's private bot chat instead of failing before upload. If the screenshot destination cannot be resolved by the screenshot bot, screenshots are skipped and the video job continues. The user must have opened the main bot with `/start` for private video delivery.


In [ ]:
#@title 2. Sync latest TMJDLC source
%cd /content
import os, subprocess
repo = '/content/TMJDLC'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'fetch', '--prune', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/Telebotfaroff/TMJDLC.git', repo], check=True)
%cd /content/TMJDLC
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'✅ TMJDLC source synced to main: {commit}')
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
#@title 3. Install dependencies (run once per fresh Colab runtime)
%cd /content/TMJDLC
import subprocess, sys
print('Installing TMJDLC and Python dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Installing FFmpeg and aria2...')
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y ffmpeg aria2'], check=True)
print('Installing Playwright Chromium and system dependencies...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'tmjdlc'], check=True)
print('✅ Dependencies installed and Python modules compiled.')


In [ ]:
#@title 4. Load credentials and configure worker performance
import os
from google.colab import userdata

SECRET_NAMES = [
    'TG_BOT_TOKEN',
    'TG_SCREENSHOT_BOT_TOKEN',
    'TG_API_ID',
    'TG_API_HASH',
    'TG_ADMIN_IDS',
    'DATABASE_URL',
]
missing = []
for key in SECRET_NAMES:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = str(value).strip()
if missing:
    raise RuntimeError('Missing Colab Secrets: ' + ', '.join(missing))

MAX_PARALLEL = 2 #@param {type:'slider', min:2, max:4, step:1}
UPLOAD_WORKERS = 8 #@param {type:'slider', min:1, max:16, step:1}
SCREENSHOT_UPLOAD_WORKERS = 2 #@param {type:'slider', min:1, max:8, step:1}
DOWNLOAD_FRAGMENTS = 8 #@param {type:'slider', min:1, max:16, step:1}
MIN_FREE_DISK_GB = 3.0 #@param {type:'number'}
AUTO_CLEAN = True #@param {type:'boolean'}
NOTIFY_PROGRESS_STEP = 10 #@param {type:'integer'}
QUEUE_POLL_SECONDS = 4 #@param {type:'integer'}
MAX_PENDING_UPLOADS = 1 #@param {type:'integer'}
UPLOAD_QUEUE_POLL_SECONDS = 2 #@param {type:'integer'}
JOB_MAX_ATTEMPTS = 3 #@param {type:'integer'}
JOB_LEASE_SECONDS = 3600 #@param {type:'integer'}

if MIN_FREE_DISK_GB < 0 or NOTIFY_PROGRESS_STEP < 1 or MAX_PENDING_UPLOADS < 0 or UPLOAD_QUEUE_POLL_SECONDS < 1:
    raise ValueError('Disk threshold must be non-negative and progress step must be at least 1.')

os.environ.update({
    'TMJDLC_MAX_PARALLEL': str(MAX_PARALLEL),
    'TMJDLC_UPLOAD_WORKERS': str(UPLOAD_WORKERS),
    'TMJDLC_SCREENSHOT_UPLOAD_WORKERS': str(SCREENSHOT_UPLOAD_WORKERS),
    'TMJDLC_DOWNLOAD_FRAGMENTS': str(DOWNLOAD_FRAGMENTS),
    'TMJDLC_USE_ARIA2': 'true',
    'TMJDLC_MIN_FREE_DISK_GB': str(MIN_FREE_DISK_GB),
    'TMJDLC_AUTO_CLEAN': str(AUTO_CLEAN).lower(),
    'TMJDLC_NOTIFY_PROGRESS_STEP': str(NOTIFY_PROGRESS_STEP),
    'TMJDLC_QUEUE_POLL_SECONDS': str(QUEUE_POLL_SECONDS),
    'TMJDLC_MAX_PENDING_UPLOADS': str(MAX_PENDING_UPLOADS),
    'TMJDLC_UPLOAD_QUEUE_POLL_SECONDS': str(UPLOAD_QUEUE_POLL_SECONDS),
    'TMJDLC_JOB_MAX_ATTEMPTS': str(JOB_MAX_ATTEMPTS),
    'TMJDLC_JOB_LEASE_SECONDS': str(JOB_LEASE_SECONDS),
    'TMJDLC_WORK_DIR': '/content/tmjdlc-bot',
})
os.makedirs(os.environ['TMJDLC_WORK_DIR'], exist_ok=True)

print('✅ Credentials loaded (values hidden).')
print(
    f'Download workers: {MAX_PARALLEL} | Video upload transmissions: {UPLOAD_WORKERS} '
    f'| Screenshot upload transmissions: {SCREENSHOT_UPLOAD_WORKERS} '
    f'| Download fragments: {DOWNLOAD_FRAGMENTS}'
)
print(f'Pause below: {MIN_FREE_DISK_GB} GiB free | Retry limit: {JOB_MAX_ATTEMPTS} | Auto-clean: {AUTO_CLEAN}')
print(f'Progress updates: every {NOTIFY_PROGRESS_STEP}% | Queue poll: {QUEUE_POLL_SECONDS}s')
print(f'Upload backlog: 1 waiting video | Download pause/resume poll: {UPLOAD_QUEUE_POLL_SECONDS}s')


In [ ]:
#@title 5. Preflight checks
import shutil, importlib.util, sys
if sys.version_info < (3, 12):
    raise RuntimeError(f'TMJDLC requires Python 3.12+. Current runtime: {sys.version.split()[0]}')
for executable in ('ffmpeg', 'ffprobe', 'aria2c'):
    if shutil.which(executable) is None:
        raise RuntimeError(f'{executable} is not installed or not on PATH.')
for module in ('pyrogram', 'psycopg', 'playwright', 'tmjdlc'):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f'Required Python module is missing: {module}')
disk = shutil.disk_usage(os.environ['TMJDLC_WORK_DIR'])
print(f'Python: {sys.version.split()[0]} | FFmpeg/FFprobe/aria2: ready')
print(f'Free Colab disk: {disk.free / 1024**3:.1f} GiB')
print('✅ Preflight passed. Neon schema initialization and expired-job recovery run at bot startup.')


## 6. Start the bot

Run the cell below and keep the Colab runtime alive while jobs are processing. The Telegram `/start` menu opens the interactive dashboard. Use its **Queue Manager** and **Settings** buttons to pause/resume the queue, cancel or retry jobs, clear completed history, configure destinations, and adjust worker concurrency.

Active jobs are allowed to finish when the queue is paused; new jobs stop being claimed. Neon stores queue state and settings, but Colab's local media files are temporary. If the runtime disconnects, unfinished jobs can be recovered from Neon, but media may need to be downloaded again.


In [ ]:
#@title 7. Start TMJDLC — Pyrogram MTProto
%cd /content/TMJDLC
print('Starting TMJDLC. Keep this cell running while the bot is in use.')
!bash -o pipefail -c 'PYTHONUNBUFFERED=1 python -u -m tmjdlc.bot 2>&1 | tee -a /content/tmjdlc-bot/bot.log'


## Telegram controls

- `/start` — open the private bot menu and interactive dashboard
- `/panel` — reopen the inline dashboard
- **Download Video** — send a Javtiful video URL; 720p is selected automatically
- **Batch Download** — send a Javtiful channel URL, then choose new/all/latest/custom pages
- **Queue Manager** — browse jobs, cancel/retry individual jobs, pause/resume, cancel queued jobs, clear completed history
- **Settings** — toggle screenshots, configure video/screenshot destinations, adjust worker concurrency
- **Storage / Worker Status** — inspect disk and worker health

Command shortcuts: `/dl VIDEO_URL`, `/batch CHANNEL_URL all`, `/batch CHANNEL_URL 1-5,8`, `/setup`, `/setupload CHANNEL_ID`, `/setimages CHANNEL_ID`, `/queue`, `/settings`, `/storage`, `/cancel JOB_ID`, `/retry JOB_ID`.

Set up the video destination using the main bot and the screenshot destination using the screenshot bot. The main bot must be an administrator in the video channel. The screenshot bot must be an administrator with posting permission in the screenshot channel. The `/setimages` setup flow verifies channel access using the screenshot bot identity.

A separate upload-only Pyrogram session (`tmjdlc_upload`) handles videos. A second Telegram bot, configured with `TG_SCREENSHOT_BOT_TOKEN`, has its own upload-only session (`tmjdlc_screenshots`) and handles screenshots. Screenshot upload concurrency is controlled by `TMJDLC_SCREENSHOT_UPLOAD_WORKERS` (default 2); video upload concurrency remains controlled by `TMJDLC_UPLOAD_WORKERS` (default 8).

The video upload client has its own peer cache, so TMJDLC copies verified channel/user access-hash records from the main session before video uploads. Screenshot uploads use the screenshot bot's own peer cache and identity. Telegram progress messages refresh about every 2 seconds. Native yt-dlp downloading is enabled by default so its progress hook can update Telegram reliably; set `TMJDLC_USE_ARIA2=true` only if you prefer aria2 over live download notifications.

### Colab storage protection and upload backpressure

TMJDLC allows one video upload at a time and at most one additional completed video waiting for the upload slot. The downloader is limited to one active download. When one upload is active, the next video can download; if that video finishes and must wait for the upload slot, the download queue stops claiming more work. It resumes automatically when an upload completes. The queue checks capacity every 2 seconds by default. The active transfer is never interrupted. Configure `TMJDLC_MAX_PENDING_UPLOADS=1` and `TMJDLC_UPLOAD_QUEUE_POLL_SECONDS=2` to retain these defaults.

### Separate screenshot bot setup

1. Create a second bot with @BotFather.
2. Save its token in Colab Secrets as `TG_SCREENSHOT_BOT_TOKEN`.
3. Add that bot as an administrator with posting permission in the screenshot destination channel.
4. Run `/setimages` and configure the screenshot channel.
5. Restart the bot after adding or changing Colab Secrets.

The main bot token and screenshot bot token must be different. The screenshot bot is used only for screenshot delivery; it does not need to handle commands or receive user messages.
